# Lab 10-5: MNIST — Dropout
### Dropout으로 과적합(overfitting) 방지
> 학습 시 keep_prob=0.7, 테스트 시 keep_prob=1.0

In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np
import random
import matplotlib.pyplot as plt
tf.set_random_seed(777)  # reproducibility

In [ ]:
# MNIST 데이터 로드 (tf.keras 방식, TF2 호환)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test  = x_test.reshape(-1, 784).astype('float32')  / 255.0

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test  = tf.keras.utils.to_categorical(y_test,  10)

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_test: ", x_test.shape,  "y_test: ", y_test.shape)

In [ ]:
# parameters
learning_rate   = 0.001
training_epochs = 15
batch_size      = 100
total_batch     = len(x_train) // batch_size

In [ ]:
X = tf.placeholder(tf.float32, [None, 784])
Y = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)  # 학습: 0.7, 테스트: 1.0

xavier = tf.keras.initializers.glorot_uniform()

W1 = tf.get_variable("W1", shape=[784, 512], initializer=xavier)
b1 = tf.Variable(tf.random_normal([512]))
L1 = tf.nn.dropout(tf.nn.relu(tf.matmul(X, W1) + b1), keep_prob=keep_prob)

W2 = tf.get_variable("W2", shape=[512, 512], initializer=xavier)
b2 = tf.Variable(tf.random_normal([512]))
L2 = tf.nn.dropout(tf.nn.relu(tf.matmul(L1, W2) + b2), keep_prob=keep_prob)

W3 = tf.get_variable("W3", shape=[512, 512], initializer=xavier)
b3 = tf.Variable(tf.random_normal([512]))
L3 = tf.nn.dropout(tf.nn.relu(tf.matmul(L2, W3) + b3), keep_prob=keep_prob)

W4 = tf.get_variable("W4", shape=[512, 512], initializer=xavier)
b4 = tf.Variable(tf.random_normal([512]))
L4 = tf.nn.dropout(tf.nn.relu(tf.matmul(L3, W4) + b4), keep_prob=keep_prob)

W5 = tf.get_variable("W5", shape=[512, 10], initializer=xavier)
b5 = tf.Variable(tf.random_normal([10]))
hypothesis = tf.matmul(L4, W5) + b5

cost      = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=hypothesis, labels=Y))
optimizer = tf.train.AdamOptimizer(learning_rate=learning_rate).minimize(cost)

In [ ]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

epoch_costs = []
for epoch in range(training_epochs):
    avg_cost = 0
    idx = np.random.permutation(len(x_train))
    x_sh, y_sh = x_train[idx], y_train[idx]

    for i in range(total_batch):
        batch_xs = x_sh[i*batch_size:(i+1)*batch_size]
        batch_ys = y_sh[i*batch_size:(i+1)*batch_size]
        c, _ = sess.run([cost, optimizer],
                        feed_dict={X: batch_xs, Y: batch_ys, keep_prob: 0.7})
        avg_cost += c / total_batch

    epoch_costs.append(avg_cost)
    print('Epoch:', '%04d' % (epoch+1), 'cost =', '{:.9f}'.format(avg_cost))

print('Learning Finished!')

In [ ]:
correct_prediction = tf.equal(tf.argmax(hypothesis, 1), tf.argmax(Y, 1))
accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
acc_val  = sess.run(accuracy, feed_dict={X: x_test, Y: y_test, keep_prob: 1})
print('Accuracy:', acc_val)

In [ ]:
# ── 학습 결과 시각화 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cost 감소 곡선
axes[0].plot(range(1, training_epochs+1), epoch_costs, 'b-o', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Avg Cost')
axes[0].set_title('Training Cost'); axes[0].grid(alpha=0.3)
for i, c in enumerate(epoch_costs):
    if i % 3 == 0:
        axes[0].annotate(f'{c:.4f}', (i+1, c), textcoords='offset points',
                         xytext=(0,6), fontsize=8, ha='center')

# 테스트 이미지 9개 + 예측
axes[1].axis('off')
fig2, ax2 = plt.subplots(3, 3, figsize=(6, 6))
fig2.suptitle(f'Random Predictions  (Accuracy: {acc_val*100:.2f}%)', fontsize=12)
for idx in range(9):
    r = random.randint(0, len(x_test)-1)
    img   = x_test[r].reshape(28, 28)
    label = np.argmax(y_test[r])
    pred  = sess.run(tf.argmax(hypothesis,1), feed_dict={X: x_test[r:r+1], keep_prob: 1})[0]
    ax2[idx//3][idx%3].imshow(img, cmap='gray')
    color = 'green' if pred==label else 'red'
    ax2[idx//3][idx%3].set_title(f'Label:{label}  Pred:{pred}', fontsize=9, color=color)
    ax2[idx//3][idx%3].axis('off')
plt.tight_layout()
plt.show()